# Roadie — recover saved fine-tuning outputs
This notebook never trains. First inspect and back up the saved artifacts in Drive.
Only run optional model evaluation/export after reconnecting to a GPU.
Do not delete the old runtime while it may still be training; inspect Drive first.
The original notebook's `status: training, trained: false` can be stale.
A complete checkpoint's `trainer_state.json` records actual completed optimizer steps.
An adapter file existing alone is not proof of a successful run; later checks verify nonzero LoRA weights.

**Start with cells 1 and 2 only.** No package installation or GPU is needed for inspection.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, datetime, zipfile
# Use your existing runtime if it reconnects. Otherwise a standard runtime suffices to inspect Drive.
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Roadie-finetuning')
RUN_PATH=''  # Optional: paste the exact ORIGINAL run folder path here if more than one exists.
runs=sorted([p for p in ROOT.iterdir() if p.is_dir()])
print('Existing runs:')
for p in runs: print(p)
if RUN_PATH:
    RUN=Path(RUN_PATH)
else:
    assert len(runs)==1, 'Multiple runs found. Set RUN_PATH to your original run directory, then rerun this cell.'
    RUN=runs[0]
assert RUN.is_dir()
print('INSPECTING:', RUN)
def read_json(path):
    return json.loads(path.read_text()) if path.exists() else None
print('Original status (may be stale):', read_json(RUN/'status.json'))
print('Root trainer step:', (read_json(RUN/'trainer-state.json') or {}).get('global_step'))
checkpoints=[]
for p in (RUN/'checkpoints').glob('checkpoint-*'):
    state=read_json(p/'trainer_state.json') or {}
    valid=(p/'adapter_config.json').exists() and (p/'adapter_model.safetensors').exists()
    checkpoints.append((int(state.get('global_step',0)),p,valid))
for step,p,valid in sorted(checkpoints):
    print(p.name,'completed steps:',step,'adapter files present:',valid,
          'weight bytes:',(p/'adapter_model.safetensors').stat().st_size if valid else None)
ADAPTER=RUN/'adapter'
if not ((ADAPTER/'adapter_model.safetensors').exists() and (ADAPTER/'adapter_config.json').exists()):
    complete=[(step,p) for step,p,valid in checkpoints if valid and step>0]
    assert complete, 'No complete saved adapter/checkpoint found. Stop and share this output.'
    ADAPTER=max(complete,key=lambda v:v[0])[1]
print('Candidate adapter:',ADAPTER)
print('Adapter configuration:',read_json(ADAPTER/'adapter_config.json'))
print('Do not infer final step count from folder existence alone.')


In [ ]:
# Back up the existing candidate plus experiment evidence; do not modify original outputs.
BACKUP=Path('/content')/(RUN.name+'-recovered-artifacts.zip')
with zipfile.ZipFile(BACKUP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in ADAPTER.rglob('*'):
        if p.is_file() and p.name not in ('optimizer.pt','scheduler.pt','rng_state.pth'):
            z.write(p,'recovered-adapter/'+str(p.relative_to(ADAPTER)))
    for p in RUN.iterdir():
        if p.is_file() and p.suffix in ('.json','.jsonl','.md','.txt'):
            z.write(p,'evidence/'+p.name)
print('Backup created:',BACKUP)
from google.colab import files
files.download(str(BACKUP))
print('STOP HERE until the download is safe. Optional cells below do not train.')


## Optional — inspect model weights and finish comparison
Continue below only with a connected GPU, after saving the ZIP.
This loads the saved adapter, uses the original evaluation inputs/settings, and writes NEW recovery files.
If the original runtime is still running, let it finish before evaluating a copy.
Install in a fresh GPU session. Restart only if Colab explicitly requests it, then rerun the inspection cell to set RUN/ADAPTER.


In [ ]:
import os, subprocess, sys
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
subprocess.run(['nvidia-smi'], check=True)
commands = [
    [sys.executable, '-m', 'pip', 'install', 'sentencepiece', 'protobuf', 'datasets==4.3.0', 'huggingface_hub>=0.34.0', 'hf_transfer'],
    [sys.executable, '-m', 'pip', 'install', '--no-deps', 'unsloth_zoo', 'bitsandbytes', 'accelerate', 'peft', 'trl', 'triton', 'unsloth'],
    ['unsloth', 'install-kernels'],
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--upgrade', 'torchao>=0.16.0'],
    [sys.executable, '-m', 'pip', 'install', 'transformers==4.57.6'],
    [sys.executable, '-m', 'pip', 'install', '--no-deps', 'trl==0.22.2'],
]
for command in commands:
    subprocess.run(command, check=True)
print('Installation complete. Continue below.')


In [ ]:
from unsloth import FastModel
import torch, time, collections, re
from contextlib import nullcontext
from safetensors import safe_open
assert torch.cuda.is_available(), 'Select a GPU before model recovery.'
RECOVERY=RUN/('recovery-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S-UTC'))
RECOVERY.mkdir(exist_ok=False)
def save_recovery(name,value):
    temp=RECOVERY/(name+'.tmp')
    temp.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=str))
    temp.replace(RECOVERY/name)
weights=ADAPTER/'adapter_model.safetensors'
nonzero_B=[]
with safe_open(str(weights),framework='pt',device='cpu') as f:
    for key in f.keys():
        if 'lora_B' in key:
            tensor=f.get_tensor(key)
            if torch.count_nonzero(tensor).item():nonzero_B.append(key)
assert nonzero_B, 'LoRA B weights are all zero; stop and investigate before claiming training.'
root_step=(read_json(RUN/'trainer-state.json') or {}).get('global_step')
checkpoint_step=(read_json(ADAPTER/'trainer_state.json') or {}).get('global_step')
CONFIG=read_json(RUN/'config.json')
assert CONFIG, 'Original configuration missing; stop before model loading.'
save_recovery('recovery-inspection.json',{'adapter_path':str(ADAPTER),
    'adapter_sha256':hashlib.sha256(weights.read_bytes()).hexdigest(),
    'nonzero_lora_B_tensors':len(nonzero_B),'root_trainer_step':root_step,
    'selected_checkpoint_step':checkpoint_step,'note':'Root adapter step is uncertain if its matching trainer-state file did not finish saving.'})
model,tokenizer=FastModel.from_pretrained(model_name=str(ADAPTER),
    max_seq_length=CONFIG['max_length'],load_in_4bit=True)
from unsloth.chat_templates import get_chat_template
tokenizer=get_chat_template(tokenizer,chat_template='gemma-3')
text_tokenizer=getattr(tokenizer,'tokenizer',tokenizer)
FastModel.for_inference(model)
EVAL_ROWS=read_json(RUN/'evaluation-inputs.json')
assert EVAL_ROWS, 'Original evaluation inputs missing; preserve the adapter and ask for help.'
def messages_for(prompt):return [{'role':'user','content':[{'type':'text','text':prompt}]}]
def compare(name,disable_adapter=False):
    results=[]
    with (model.disable_adapter() if disable_adapter else nullcontext()):
        for row in EVAL_ROWS:
            inputs=tokenizer.apply_chat_template(messages_for(row['prompt']),tokenize=True,
                add_generation_prompt=True,return_tensors='pt',return_dict=True).to('cuda')
            n=inputs['input_ids'].shape[-1]
            assert n+CONFIG['max_new_tokens']<=CONFIG['max_length']
            started=time.monotonic()
            with torch.inference_mode():
                output=model.generate(**inputs,do_sample=False,max_new_tokens=CONFIG['max_new_tokens'],use_cache=True)
            tokens=output[0,n:]
            answer=tokenizer.decode(tokens,skip_special_tokens=True)
            results.append({'id':row['id'],'split':row['split'],'question':row['question'],
                'evidence':row['evidence'],'reference':row.get('original_answer'),'answer':answer,
                'new_tokens':len(tokens),'reached_token_cap':len(tokens)>=CONFIG['max_new_tokens'],
                'seconds':time.monotonic()-started,'contains_S1':'[S1]' in answer})
            save_recovery(name+'-answers.json',results)
            print(name,row['id'],'tokens:',len(tokens))
            del inputs,output
    return results
# Recompute both conditions from the same recovered model to avoid mixing runtime comparisons.
base_answers=compare('base',disable_adapter=True)
tuned_answers=compare('tuned')
report=['# Recovered Roadie fine-tuning comparison',
        'Saved adapter verified to have nonzero LoRA B tensors. Human answer review pending.',
        'Root trainer step: '+str(root_step)+'; selected checkpoint step: '+str(checkpoint_step),
        'This compares adapter disabled versus enabled on the same recovered base. No new training.']
pairs=[]
for a,b in zip(base_answers,tuned_answers):
    assert a['id']==b['id']
    pairs.append({'id':a['id'],'question':a['question'],'evidence':a['evidence'],
                  'base':a['answer'],'tuned':b['answer'],'review':'pending'})
    report+=['## '+a['id'],a['question'],'### Evidence',a['evidence'],
             '### Base',a['answer'],'### Tuned',b['answer'],
             'Token cap flags: '+str(a['reached_token_cap'])+', '+str(b['reached_token_cap'])]
save_recovery('paired-review.json',pairs)
(RECOVERY/'comparison.md').write_text('\n\n'.join(report))
print('Saved comparisons to:',RECOVERY)
print('Download comparison.md for review. No new training has run.')


## Optional export — off by default
Only enable after safeguarding the adapter and reviewing the saved comparison.
This follows the official Gemma 3 Q8_0 export route. Q4 conversion and tested Ollama import still follow separately.
Export needs substantial additional memory, disk and time; keep the original adapter safe.


In [ ]:
EXPORT_GGUF=False
if EXPORT_GGUF:
    import shutil
    assert shutil.disk_usage('/content').free>25*1024**3, 'Need 25 GiB free for export intermediates.'
    export=Path('/content/roadie-recovered-gguf')
    model.save_pretrained_gguf(str(export),tokenizer,quantization_method='Q8_0')
    shutil.copytree(export,RECOVERY/'gguf-export',dirs_exist_ok=True)
    print('GGUF copied to:',RECOVERY/'gguf-export')
else:
    print('Export skipped. Original adapter/checkpoints are untouched.')
